In [1]:
# Deeper CNN Architecture (VGG-inspired)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import io
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split

# --- CNN Architecture ---
class VggCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(VggCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [2]:
from torch.utils.data import Dataset
from PIL import Image
import io
from sqlalchemy import create_engine, text
import torchvision.transforms as transforms

# --- Dataset Class with Train/Val Split ---
class TrailSegmentDataset(Dataset):
    def __init__(self, db_url, transform=None, mode='train'):
        self.engine = create_engine(db_url)
        self.transform = transform

        # Load and split data
        with self.engine.connect() as conn:
            result = conn.execute(text("""
                SELECT id, surface_group FROM trail_segments 
                WHERE image_data IS NOT NULL AND surface_group IS NOT NULL
            """))
            all_records = result.fetchall()

        self.label_map = self._build_label_map([r[1] for r in all_records])
        all_records = [(r[0], self.label_map[r[1]]) for r in all_records]

        train_records, val_records = train_test_split(all_records, test_size=0.2, random_state=42)
        self.records = train_records if mode == 'train' else val_records

    def _build_label_map(self, labels):
        unique_labels = sorted(set(labels))
        return {label: idx for idx, label in enumerate(unique_labels)}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        seg_id, label = self.records[idx]

        with self.engine.connect() as conn:
            result = conn.execute(text("SELECT image_data FROM trail_segments WHERE id = :id"), {"id": seg_id})
            image_bytes = result.scalar()

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

db_url = "postgresql://markfinlay:@localhost:5432/osm"
train_dataset = TrailSegmentDataset(db_url=db_url, transform=transform, mode='train')
val_dataset = TrailSegmentDataset(db_url=db_url, transform=transform, mode='val')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Label mapping:", train_dataset.label_map)

Label mapping: {'asphalt': 0, 'grass': 1, 'gravel': 2, 'mud/dirt': 3, 'paved': 4}


In [4]:
from collections import Counter
print(Counter([label for _, label in train_dataset.records]))

Counter({3: 5803, 1: 5655, 2: 5632, 0: 5158, 4: 1679})


In [5]:
import torch
from collections import Counter
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

counts = Counter([label for _, label in train_dataset.records])
total = sum(counts.values())
num_classes = len(counts)

# Inverse frequency: weight = total / (num_classes * count)
weights = [total / (num_classes * counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

for i, w in enumerate(weights):
    print(f"Class {i}: weight = {w:.4f}")

Class 0: weight = 0.9278
Class 1: weight = 0.8462
Class 2: weight = 0.8497
Class 3: weight = 0.8246
Class 4: weight = 2.8501


In [6]:
model = VggCNN(num_classes=len(train_dataset.label_map)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [7]:
# --- Training Loop with Validation ---
epoch_count = 50
for epoch in range(epoch_count):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/50] - Training Loss: {running_loss:.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_accuracy = 100 * correct / total
    print(f"→ Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.2f}%")

print("Training complete.")

Epoch [1/50] - Training Loss: 1199.1835
→ Validation Loss: 304.8794, Accuracy: 26.11%
Epoch [2/50] - Training Loss: 1166.4141
→ Validation Loss: 294.6399, Accuracy: 23.74%
Epoch [3/50] - Training Loss: 1104.9007
→ Validation Loss: 258.0107, Accuracy: 37.30%
Epoch [4/50] - Training Loss: 1017.0096
→ Validation Loss: 248.8545, Accuracy: 42.09%
Epoch [5/50] - Training Loss: 991.0854
→ Validation Loss: 249.7822, Accuracy: 40.66%
Epoch [6/50] - Training Loss: 983.5129
→ Validation Loss: 244.3462, Accuracy: 41.46%
Epoch [7/50] - Training Loss: 975.8870
→ Validation Loss: 243.3906, Accuracy: 43.81%
Epoch [8/50] - Training Loss: 968.6713
→ Validation Loss: 242.0526, Accuracy: 44.47%
Epoch [9/50] - Training Loss: 963.0702
→ Validation Loss: 240.6975, Accuracy: 42.86%
Epoch [10/50] - Training Loss: 960.4993
→ Validation Loss: 240.8621, Accuracy: 44.10%
Epoch [11/50] - Training Loss: 959.3392
→ Validation Loss: 243.2846, Accuracy: 41.79%
Epoch [12/50] - Training Loss: 953.4703
→ Validation Loss: 

In [ ]:
# Look at the Framework 
# Random Seed? - Set the random seed value
# Heurstic - test nuking?
# Reread the Kenya paper